In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import re, json, io
import pandas as pd
import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
text = """Apple Inc. announced on 12 September 2025 that Tim Cook will open a new office in Bengaluru, India.
The company will invest $500 million over the next 3 years. Contact the press team at press@apple.com
or call +91 98765 43210. Sundar Pichai of Google met Narendra Modi in New Delhi on 3 March 2025."""
doc = nlp(text)

In [ ]:
for ent in doc.ents:
    print(f"{ent.text:25s} -> {ent.label_:10s} ({spacy.explain(ent.label_)})")
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
patterns = {
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d{2}[\s-]?\d{5}[\s-]?\d{5}",
    "date":  r"\b\d{1,2}\s(?:January|February|March|April|May|June|July|August|September|October|November|December)\s\d{4}\b",
    "money": r"\$\d[\d,]*(?:\.\d+)?\s?(?:million|billion)?",
}
for name, pat in patterns.items():
    print(f"{name:6s}:", re.findall(pat, text))

In [ ]:
def svo(doc):
    triples = []
    for tok in doc:
        if tok.pos_ == "VERB":
            subj = [w for w in tok.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w for w in tok.rights if w.dep_ in ("dobj", "attr")]
            for s in subj:
                for o in obj:
                    triples.append((" ".join(t.text for t in s.subtree), tok.lemma_, " ".join(t.text for t in o.subtree)))
    return triples

for t in svo(doc): print(t)

In [ ]:
record = {
    "organizations": sorted({e.text for e in doc.ents if e.label_ == "ORG"}),
    "people": sorted({e.text for e in doc.ents if e.label_ == "PERSON"}),
    "locations": sorted({e.text for e in doc.ents if e.label_ in ("GPE", "LOC")}),
    "dates": re.findall(patterns["date"], text),
    "emails": re.findall(patterns["email"], text),
    "money": re.findall(patterns["money"], text),
}
print(json.dumps(record, indent=2))

In [ ]:
csv_data = """id,name,email,joined,salary
1,Asha Rao,asha@example.com,2023-04-12,55000
2,Ravi Kumar,ravi@example.org,2022-11-03,62000
3,Meena Iyer,meena@example.com,2024-01-20,58000
"""
df = pd.read_csv(io.StringIO(csv_data), parse_dates=["joined"])
display(df)

print("Employees joined after 2023:", df[df.joined > "2023-01-01"].name.tolist())
print("Average salary :", df.salary.mean())
print("Email domains  :", df.email.str.extract(r"@(.+)$")[0].value_counts().to_dict())

js = json.loads('{"order": {"id": 101, "customer": {"name": "Ravi", "city": "Lucknow"}, "items": [{"sku": "A1", "qty": 2}, {"sku": "B7", "qty": 1}]}}')
print("\nJSON extraction ->", js["order"]["customer"]["name"], js["order"]["customer"]["city"],
      sum(i["qty"] for i in js["order"]["items"]), "items")